### data ingestion

In [31]:
from langchain_core.documents import Document

In [32]:
doc= Document(
    page_content="this is the main page content",
    metadata={
        "source":"xysz.txt",
        "authors": "suryo",
        "page":2,
        "date created":"24-01-26"

    }
)

In [33]:
doc

Document(metadata={'source': 'xysz.txt', 'authors': 'suryo', 'page': 2, 'date created': '24-01-26'}, page_content='this is the main page content')

In [34]:
from langchain_community.document_loaders import TextLoader
loader= TextLoader(r"C:\Users\KIIT\Desktop\RAG\data\chatgpt.txt", encoding="utf-8")
loader

In [35]:
document=loader.load()
print(document)

[Document(metadata={'source': 'C:\\Users\\KIIT\\Desktop\\RAG\\data\\chatgpt.txt'}, page_content='')]


In [45]:
## directory loader
from langchain_community.document_loaders import DirectoryLoader
dir_loader= DirectoryLoader(
    r"C:\Users\KIIT\Desktop\RAG\data",
    glob="**/*.txt",
    loader_cls= TextLoader,
    show_progress=False
)
documents= dir_loader.load(

)
documents

[Document(metadata={'source': 'C:\\Users\\KIIT\\Desktop\\RAG\\data\\chatgpt.txt'}, page_content='')]

In [46]:
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader

dir_loader=DirectoryLoader(
    r"C:\Users\KIIT\Desktop\RAG\data\pdfs_ml",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)
pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'Adobe InDesign 18.0 (Windows)', 'creationdate': '2012-08-24T16:03:10+05:30', 'source': 'C:\\Users\\KIIT\\Desktop\\RAG\\data\\pdfs_ml\\28th-KES-International-Conference-on-Knowledge-Based-and-I_2024_Procedia-Com.pdf', 'file_path': 'C:\\Users\\KIIT\\Desktop\\RAG\\data\\pdfs_ml\\28th-KES-International-Conference-on-Knowledge-Based-and-I_2024_Procedia-Com.pdf', 'total_pages': 9, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2013-02-20T16:20:50+05:30', 'trapped': '', 'modDate': "D:20130220162050+05'30'", 'creationDate': "D:20120824160310+05'30'", 'page': 0}, page_content='ScienceDirect\nAvailable online at www.sciencedirect.com\nProcedia Computer Science 246 (2024) 1–9\n1877-0509 © 2024 The Authors. Published by Elsevier B.V.\nThis is an open access article under the CC BY-NC-ND license (https://creativecommons.org/licenses/by-nc-nd/4.0)\nPeer-review under responsibi

In [47]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [48]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} pdf files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

In [49]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(pdf_documents)
print(f"total chunks created are {len(chunks)}")
print(chunks[0].page_content[:500])

total chunks created are 4430
ScienceDirect
Available online at www.sciencedirect.com
Procedia Computer Science 246 (2024) 1–9
1877-0509 © 2024 The Authors. Published by Elsevier B.V.
This is an open access article under the CC BY-NC-ND license (https://creativecommons.org/licenses/by-nc-nd/4.0)
Peer-review under responsibility of the scientific committee of the 28th International Conference on Knowledge 
Based and Intelligent information and Engineering Systems
10.1016/j.procs.2024.09.152
© 2024 The Authors. Published by El


### embedding


In [54]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

### storing vecs


In [61]:
from langchain_community.vectorstores import Chroma
from tqdm import tqdm

batch_size = 3000  
vectorstore = Chroma(persist_directory="./chroma_db")
for i in tqdm (range(0,len(chunks),batch_size), desc="embedding batches"):
    batch_chunks=chunks[i:i+batch_size]
    batch_vectors =embeddings.embed_documents([c.page_content for c in batch_chunks])
    vectorstore.add_documents(documents=batch_chunks,embeddings=batch_vectors)
    vectorstore.persist()

embedding batches:   0%|          | 0/2 [00:00<?, ?it/s]C:\Users\KIIT\AppData\Local\Temp\ipykernel_21584\1824499236.py:10: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()
embedding batches: 100%|██████████| 2/2 [03:37<00:00, 108.99s/it]


In [62]:
vectorestore= Chroma(persist_directory="./chroma_db")

In [63]:
retriver = vectorstore.as_retriever(search_kwargs={"k": 4})
results= retriver._get_relevant_documents("logistic regression", run_manager=None)

#Deduplicate chunks
seen_texts = set() #keeps track of chunks you’ve already added
unique_results = []

for r in results:
    # create a key based on source file + content
    content_key = r.page_content.strip() #makes sure minor whitespace differences don’t create “new” chunks
    if content_key not in seen_texts:
        unique_results.append(r)
        seen_texts.add(content_key)
for r in unique_results:
    print("source:", r.metadata["source_file"])
    print(r.page_content[:400])
    print("---------")

source: Can-Generative-AI-be-used-to-improve-doctor-patient_2024_Procedia-Computer-S.pdf
associated costs, Art Int Med, 68, 59-69, doi: 10.1016/j.artmed.2016.03.001. 
[23] Belciug, S., (2020). Logistic regression paradigm for training a single-hidden layer feedforward neural network. Application to gene expression 
datasets for cancer research. J Biomed Inf, 102, 103373, doi: 10.1016/j.jbi.2019.103373. 
[24] Cer, D. et al. (2018) Universal Sentence Encoder. arXiv:1803.11175
---------


In [58]:
from langchain_classic.chains import RetrievalQA
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [59]:
from huggingface_hub import snapshot_download
from tqdm import tqdm

snapshot_download(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    local_dir="./tinyllama",
    tqdm_class=tqdm
)

print("Download complete!")

Fetching 10 files: 100%|██████████| 10/10 [00:00<?, ?it/s]

Download complete!


In [65]:


model_id = "./tinyllama"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1, # Lower temp = more factual for RAG
    repetition_penalty=1.1
)
llm = HuggingFacePipeline(pipeline=pipe)

# --- STEP 3: THE RAG CHAIN ---
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}), # Top 4 chunks
    return_source_documents=True
)

# Run the test
query = "what is k nearest"
result = qa.invoke({"query": query})

print("\n--- ANSWER ---")
print(result["result"])

Device set to use cpu



--- ANSWER ---
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

2. Related work
Research in machine learning has extensively focused on improving clustering algorithms, with special attention
to k-means. This area is crucial for advancing data classification and pattern recognition, highlighting the importance
of improving these algorithms. A significant part of this enhancement involves the exploration of different distance
metrics and aggregation methods to improve the algorithms’ precision and efficiency [1,14].
k-means, known for its straightforward approach and effectiveness in processing large data sets, traditionally uses
Euclidean distance to determine how close data points are to cluster centroids. However, this measurement may not
always accurately reflect the data’s structure, especially in complex, high-dimensional environments. To address this
issue, recent r